In [ ]:
#SETUP DOCKER+LANGFUSE

#!cd ~/Desktop/Mini-Project-LangFuse/langfuse
#!docker compose up -d
#http://localhost:3000/

In [ ]:
#SETUP LLM+LANGFUSE

#Import keys
import os
import json
from dotenv import load_dotenv
load_dotenv()

#Set up the LLM client
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY_1"),
    base_url="https://api.groq.com/openai/v1"
)

#Connect LangFuse
from langfuse import observe, Langfuse, get_client
langfuse = Langfuse()

#Models
BASELINE_MODEL = "llama-3.1-8b-instant"
NEW_MODEL = "llama-3.3-70b-versatile"

In [ ]:
#SIMPLE CHATBOT

#Prompt
SYSTEM_PROMPT = """You are a helpful healthcare assistant. You provide accurate, 
general health information to help users understand medical topics.

Rules:
- Always recommend consulting a doctor for specific medical advice.
- Be clear and concise.
- If you don't know something, say so honestly.
- Never diagnose conditions or prescribe medication.
"""

#ChatBot
@observe()
def healthcare_chatbot(user_question, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
    )
    return response.choices[0].message.content

In [ ]:
#LLM AS A JUDGE

#Prompt
JUDGE_PROMPT = """You are an expert evaluator for a healthcare chatbot. 
Compare the actual answer to the expected answer and score the actual answer.

Question: {question}

Expected Answer: {expected_answer}

Actual Answer: {actual_answer}

Score the actual answer from 1 to 5:
1 = Completely wrong or harmful
2 = Mostly incorrect or missing key information
3 = Partially correct but incomplete
4 = Mostly correct with minor omissions
5 = Fully correct and complete

Respond with ONLY a JSON object in this exact format, nothing else:
{{"score": <number>, "reasoning": "<brief explanation>"}}
"""

#Judge
@observe()
def judge_answer(question, expected_answer, actual_answer, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": JUDGE_PROMPT.format(
                question=question,
                expected_answer=expected_answer,
                actual_answer=actual_answer
            )}
        ]
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
#GOLDEN DATASET (from Claude)

#Open Json dataset
with open("golden_dataset.json", "r") as f:
    golden_dataset = json.load(f)

print(f"Loaded {len(golden_dataset)} questions")

#Create the Dataset in LangFuse
dataset = langfuse.create_dataset(name="healthcare-golden-dataset")
print(f"Dataset created: {dataset.name}")

#Upload each Q&A pair into LangFuse
for item in golden_dataset:
    langfuse.create_dataset_item(
        dataset_name="healthcare-golden-dataset",
        input=item["question"],
        expected_output=item["expected_answer"],
        metadata={"id": item["id"], "category": item["category"]}
    )
    print(f"  Uploaded Q{item['id']}: {item['question'][:50]}...")

In [ ]:
#RUN EVALUATION (via LangFuse Dataset)

def run_langfuse_evaluation(model_name):
    dataset = langfuse.get_dataset("healthcare-golden-dataset")
    results = []
    
    print(f"\nEvaluating: {model_name}")
    print(f"{'='*50}")
    
    for item in dataset.items:
        print(f"Running: {item.input[:50]}...")
        
        with item.run(run_name=f"eval-{model_name}") as span:
            answer = healthcare_chatbot(item.input, model=model_name)
            eval_result = judge_answer(item.input, item.expected_output, answer)

            #Update span
            span.update(input=item.input, output=answer)

            #Log score
            trace_id = get_client().get_current_trace_id()
            if trace_id:
                langfuse.create_score(
                    trace_id=trace_id,
                    name="judge-score",
                    value=eval_result["score"],
                    comment=eval_result["reasoning"]
                )

            #Custome table
            results.append({
                "id": item.metadata.get("id", "?"),
                "category": item.metadata.get("category", "unknown"),
                "question": item.input,
                "score": eval_result["score"],
                "reasoning": eval_result["reasoning"]
            })
            
            print(f"  Score: {eval_result['score']}")

    #Average score
    avg_score = sum(r["score"] for r in results) / len(results)
    print(f"\nAverage Score: {avg_score:.2f} / 5.00")
    
    return results, avg_score

run_langfuse_evaluation(BASELINE_MODEL)
run_langfuse_evaluation(NEW_MODEL)